In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
import joblib

# ---------------------------------------------------------
# Robust project root detection
# ---------------------------------------------------------
def find_project_root(start_path: Path) -> Path:
    for parent in [start_path] + list(start_path.parents):
        if (parent / "data").exists() and (parent / "scripts").exists():
            return parent
    raise RuntimeError("Project root not found.")

PROJECT_ROOT = find_project_root(Path.cwd())

DATA_LABELS_DIR = PROJECT_ROOT / "data" / "labels"
MODEL_DIR = PROJECT_ROOT / "models" / "lgbm"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Labels dir:", DATA_LABELS_DIR)
print("Model dir:", MODEL_DIR)


FileNotFoundError: Could not find module 'c:\Trading\Projects\ES_AI_Project\es_env\Lib\site-packages\lightgbm\bin\lib_lightgbm.dll' (or one of its dependencies). Try using the full path with constructor syntax.

In [ ]:
# Load labeled dataset
df = pd.read_csv(DATA_LABELS_DIR / "labeled.csv")

# Target column
TARGET = "ShortSuccess"

# Features = all numeric columns except target
FEATURE_COLS = [c for c in df.columns if c != TARGET]

X = df[FEATURE_COLS].astype(np.float32)
y = df[TARGET].astype(np.int8)

print("Rows:", len(df))
print("Features:", len(FEATURE_COLS))
print("Target distribution:")
print(y.value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))


In [ ]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)


In [ ]:
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_data_in_leaf": 50,
    "verbose": -1
}

params


In [ ]:
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    num_boost_round=2000,
    early_stopping_rounds=100
)

print("Best iteration:", model.best_iteration)


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score

val_pred_prob = model.predict(X_val)
val_pred = (val_pred_prob > 0.5).astype(int)

auc = roc_auc_score(y_val, val_pred_prob)
acc = accuracy_score(y_val, val_pred)

print("Validation AUC:", auc)
print("Validation Accuracy:", acc)


In [ ]:
# Save LightGBM model
model.save_model(str(MODEL_DIR / "lgbm_model.txt"))

# Save feature list
with open(MODEL_DIR / "feature_list.txt", "w") as f:
    for col in FEATURE_COLS:
        f.write(col + "\n")

print("Model and feature list saved.")


In [ ]:
print("Training complete. Model saved to:")
print(MODEL_DIR / "lgbm_model.txt")
